In [1]:
# ===========================
# 01. IMPORTS
# ===========================

from delta.tables import DeltaTable
from pyspark.sql import types
import json


# Perform the Delta MERGE operation for each micro-batch
def upsert(df, delta_table):

    # Create a temporary global view from the current micro-batch
    df.createOrReplaceGlobalTempView(f"view_{table}")

    # Keep only the latest record for each primary key
    query = f"""
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY {pk_field}
                       ORDER BY op_ts DESC
                   ) AS rn
            FROM global_temp.view_{table}
        ) t
        WHERE rn = 1
    """

    # Remove auxiliary columns before the MERGE operation
    control_cols = ["rn"]
    df_cdc = spark.sql(query).drop(*control_cols)

    # Apply CDC changes to the target Delta table
    (
        delta_table.alias("b")
        .merge(df_cdc.alias("d"), f"b.{pk_field} = d.{pk_field}")
        .whenMatchedDelete(condition="d.op_type = 'D'")
        .whenMatchedUpdateAll(condition="d.op_type = 'U'")
        .whenNotMatchedInsertAll(condition="d.op_type IN ('I', 'U')")
        .execute()
    )


# Load the table schema from a JSON file
def import_schema(tablename):

    with open(
        f"/Workspace/ai-data-platform/src/table-schema/{tablename}.json",
        "r"
    ) as open_file:
        schema_json = json.load(open_file)

    return types.StructType.fromJson(schema_json)

In [35]:
# ===========================
# 02. PARAMETERS
# ===========================

# Default values
DEFAULT_CATALOG = "lab_catalog_bronze"
DEFAULT_SCHEMA = "digital_insurance"
DEFAULT_TABLE = "tb_apolice"
DEFAULT_PK_FIELD = "id"

# Read parameters with fallback values
catalog = oidlUtils.parameters.getParameter("catalog", DEFAULT_CATALOG)
schema = oidlUtils.parameters.getParameter("schema", DEFAULT_SCHEMA)
table = oidlUtils.parameters.getParameter("table", DEFAULT_TABLE)
pk_field = oidlUtils.parameters.getParameter("pk_field", DEFAULT_PK_FIELD)

# Derived values
full_table_name = f"{catalog}.{schema}.{table}"
source_folder = f"/Volumes/lab_catalog_raw/digital_insurance/FNOL/USER_ADMIN/{table.upper()}"
volume_path = f"{source_folder}/*.parquet"
checkpoint_path = f"{source_folder}_checkpoint/"

# Load the table schema from the JSON definition
table_schema = import_schema(table)

In [ ]:
# ===========================
# 03. FULL LOAD INGESTION
# ===========================

# Create the Delta table only if it does not already exist
if not spark.catalog.tableExists(full_table_name):
    print("Target table does not exist. Starting full load...")

    # Read all Parquet files and create a temporary view
    (
        spark.read
             .parquet(volume_path)
             .createOrReplaceTempView(f"view_{table}")
    )

    # Keep only the latest record for each primary key
    query_full = f"""
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY {pk_field}
                       ORDER BY op_ts DESC
                   ) AS rn
            FROM view_{table}
        ) t
        WHERE rn = 1
          AND op_type <> 'D'
    """

    # Remove control columns before writing the Delta table
    control_cols = ["current_ts", "pos", "rn"]
    df_query_full = spark.sql(query_full).drop(*control_cols)

    # Create the target Delta table
    (
        df_query_full.write
                     .format("delta")
                     .mode("overwrite")
                     .saveAsTable(full_table_name)
    )

    print("Full load completed successfully.")

else:
    print("Target table already exists. Skipping full load.")

In [ ]:
# ===========================
# 04. INCREMENTAL CDC PROCESSING
# ===========================

# Get a reference to the target Delta table
delta_table_bronze = DeltaTable.forName(spark, full_table_name)

# Create a streaming DataFrame from the Parquet files
df_stream = (
    spark.readStream
         .format("parquet")
         .schema(table_schema)
         .load(volume_path)
)

# Process each micro-batch and perform the Delta MERGE operation
query = (
    df_stream.writeStream
             .option("checkpointLocation", checkpoint_path)
             .foreachBatch(lambda df, batch_id: upsert(df, delta_table_bronze))
             .trigger(availableNow=True)
             .start()
)

# Wait until all available data has been processed
query.awaitTermination()